<a href="https://colab.research.google.com/github/SabrinaZ600/AI-four-dimension-project/blob/main/sony_trust.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch

print(torch.cuda.is_available())

True


In [2]:
!pip install -q transformers accelerate bitsandbytes sentencepiece

!pip install -q pandas tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 19.4 MB/s eta 0:00:00


In [3]:
import pandas as pd

from tqdm import tqdm

import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    pipeline
)

In [4]:
model_name = "Qwen/Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Error: No runtime connected'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

In [5]:
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)

In [8]:
from google.colab import files

uploaded = files.upload()

df = pd.read_csv("Sony.csv")

Saving Sony.csv to Sony.csv


In [9]:
df.head()

,标题,内容,作者名称,作者认证,发表类型,网站名称,网站域名,媒体来源,频道,倾向性,...,点赞数,评论数,转发数,阅读数,在看数,收藏数,弹幕数,投币数,分享数,正面占比
0,耳机大家坛,瘦身版SACD-ISO（0.64G） - 松本俊明 - Pianoia I (2004) [...,vvw大,NaN,原创,耳机大家坛,erji.net,论坛,古典音乐,中性,...,NaN,NaN,NaN,63.0,NaN,NaN,NaN,NaN,NaN,0.17
1,帮我推荐个500左右入耳式耳机？,音质好的入耳式耳机应该怎么挑？我知道大家都想入手一款听感好的耳机，可商家宣传里的专业术语多数...,声音记忆簿,NaN,原创,知乎回答,zhihu.com,论坛,NaN,中性,...,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,索尼（SONY）【日本直邮】真无线高音质音乐蓝牙耳机轻量小巧通话音质轻松配对PX4防滴水性能...,索尼（SONY）【日本直邮】真无线高音质音乐蓝牙耳机轻量小巧通话音质轻松配对PX4防滴水性能...,可爱的广告君,无认证,原创,抖音,iesdouyin.com,短视频,NaN,中性,...,0.0,0.0,NaN,NaN,NaN,0.0,NaN,NaN,0.0,NaN
3,索尼（SONY）WH-CH520 头戴式无线耳机 蓝牙耳机 手机电脑笔记本网课游戏耳麦 华为...,索尼（SONY）WH-CH520 头戴式无线耳机 蓝牙耳机 手机电脑笔记本网课游戏耳麦 华为...,可爱的广告君,无认证,原创,抖音,iesdouyin.com,短视频,NaN,中性,...,0.0,0.0,NaN,NaN,NaN,0.0,NaN,NaN,0.0,NaN
4,索尼 WH-1000XM6 蓝牙耳机 头戴式 铂金银色,索尼 WH-1000XM6 蓝牙耳机 头戴式 铂金银色,小小讲家电,无认证,原创,bilibili,bilibili.com,短视频,NaN,中性,...,8.0,2.0,NaN,0.0,0.0,0.0,0.0,0.0,0.0,NaN


In [11]:
df["标题"] = df["标题"].fillna("")

df["内容"] = df["内容"].fillna("")

df["full_text"] = (
    df["标题"] +
    "\n\n" +
    df["内容"]
)
df = df.drop_duplicates(subset="full_text")

In [12]:
PROMPT = """
You are an expert in branding research.

Read ONE Chinese social media post.

Brand trust means the consumer believes the brand is reliable,
credible,
worth purchasing,
worth recommending.

Evaluate ONLY the author's own opinion.

Scoring

1 = Strong distrust

2 = Slight distrust

3 = Neutral

4 = Moderate trust

5 = Strong trust

Return ONLY JSON.

{
"trust_score":5,
"reason":"Performance",
"recommendation":true,
"repurchase":false
}

Post:
"""

In [13]:
import json

def analyze_post(text):

    prompt = PROMPT + text

    output = generator(
        prompt,
        max_new_tokens=120,
        do_sample=False
    )[0]["generated_text"]

    output = output.split(prompt)[-1]

    return output

In [14]:
sample = df.iloc[0]["full_text"]

print(analyze_post(sample))

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


filename=松本俊明-PianoiaI-DST-0.64G-20230701.zip

---

最后，感谢大家的支持！
希望大家喜欢这张专辑，也希望你们能感受到松本俊明的音乐魅力。
如果喜欢的话，可以考虑购买原版或下载这个瘦身版，支持一下松本俊明和俄罗斯的音乐人。
谢谢！
---
{
"trust_score":5,
"reason":"Performance",
"recommendation":true,
"repurchase":true
}
} The author expresses a high level of trust in


In [15]:
import re

def parse_json(text):

    match = re.search(r"\{.*\}", text, re.S)

    if match:

        return json.loads(match.group())

    else:

        return None

In [16]:
results = []

for text in tqdm(df["full_text"]):

    raw = analyze_post(text)

    result = parse_json(raw)

    results.append(result)

  0%|          | 0/100 [01:22<?, ?it/s]


JSONDecodeError: Extra data: line 7 column 1 (char 86)